# Final Project — Merge: Operations & Environmental
**Business 35137 — Winter 2026**

In [1]:
import pandas as pd
import numpy as np
import os

DATA_DIR         = '../Data/'
intermediate_dir = os.path.join(DATA_DIR, 'Intermediate')

In [2]:
import pgeocode

# ── State abbreviation → 2-digit Census FIPS ─────────────────────────────────
STATE_FIPS = {
    'AL':'01','AK':'02','AZ':'04','AR':'05','CA':'06','CO':'08','CT':'09',
    'DE':'10','DC':'11','FL':'12','GA':'13','HI':'15','ID':'16','IL':'17',
    'IN':'18','IA':'19','KS':'20','KY':'21','LA':'22','ME':'23','MD':'24',
    'MA':'25','MI':'26','MN':'27','MS':'28','MO':'29','MT':'30','NE':'31',
    'NV':'32','NH':'33','NJ':'34','NM':'35','NY':'36','NC':'37','ND':'38',
    'OH':'39','OK':'40','OR':'41','PA':'42','RI':'44','SC':'45','SD':'46',
    'TN':'47','TX':'48','UT':'49','VT':'50','VA':'51','WA':'53','WV':'54',
    'WI':'55','WY':'56',
}

# ── Build (city, state_abbr) → county_fips lookup via pgeocode ────────────────
# pgeocode US data: place_name, state_code (abbr), county_code (3-digit FIPS).
# Multiple ZIP codes share a city; take the modal county per (city, state).
nomi = pgeocode.Nominatim('us')
geo  = (nomi._data[['place_name', 'state_code', 'county_code']]
              .dropna(subset=['place_name', 'state_code', 'county_code'])
              .copy())

geo['city_key']    = geo['place_name'].str.strip().str.lower()
geo['state_fips']  = geo['state_code'].map(STATE_FIPS)
geo['county3']     = geo['county_code'].astype(int).astype(str).str.zfill(3)
geo['county_fips'] = (geo['state_fips'] + geo['county3']).str.zfill(5)  # guard: always 5 chars

city_county: dict[tuple, str] = (
    geo.dropna(subset=['county_fips'])
       .groupby(['city_key', 'state_code'])['county_fips']
       .agg(lambda x: x.value_counts().index[0])   # modal county
       .to_dict()
)

# ── Format check ──────────────────────────────────────────────────────────────
fips_lens = pd.Series(list(city_county.values())).str.len().value_counts()
print(f"City-county lookup: {len(city_county):,} (city, state) pairs")
print(f"county_fips length distribution (all should be 5): {fips_lens.to_dict()}")
print(f"Sample FIPS: {list(city_county.values())[:5]}")


City-county lookup: 29,540 (city, state) pairs
county_fips length distribution (all should be 5): {5: 29540}
Sample FIPS: ['42027', '01067', '13315', '22113', '28071']


In [3]:
master_crsp = pd.read_parquet(os.path.join(intermediate_dir, 'master_crsp_merged.parquet'))

# ── Filter to Russell 2000 / S&P 500 universe ─────────────────────────────────
universe = master_crsp[master_crsp['index'].isin(['Russell 2000', 'S&P 500'])]

# ── Filter to atmospheric sectors only ────────────────────────────────────────
ATMOS_SECTORS = ['Agriculture & Food', 'Utilities & Telecom', 'Leisure & Tourism']
universe = universe[universe['atmos_sector'].isin(ATMOS_SECTORS)].copy()

print(f"Sector breakdown:\n{universe['atmos_sector'].value_counts()}")
print(f"\nUniverse: {universe['permno'].nunique():,} unique permnos")
print(f"  Russell 2000 rows: {(universe['index'] == 'Russell 2000').sum():,}")
print(f"  S&P 500 rows:      {(universe['index'] == 'S&P 500').sum():,}")

Sector breakdown:
atmos_sector
Utilities & Telecom    106993
Agriculture & Food      56943
Leisure & Tourism       51406
Name: count, dtype: int64

Universe: 2,087 unique permnos
  Russell 2000 rows: 135,013
  S&P 500 rows:      80,329


In [4]:
# ── Load company operations locations from 10-K scan ─────────────────────────
ops = pd.read_csv(os.path.join(intermediate_dir, 'Industry_SP_Russell_Company_Operations.csv'))

# Rename to avoid collision with CRSP-native state/addzip columns
ops = ops.rename(columns={'State': 'operations_state', 'Zip': 'operations_zip'})

# ── Merge universe with operations on gvkey ───────────────────────────────────
# Zero-pad to 6 digits — CRSP stores gvkey as "012994", ops saves raw integers
universe['gvkey'] = universe['gvkey'].astype(str).str.zfill(6)
ops['gvkey']      = ops['gvkey'].astype(str).str.zfill(6)

merged = universe.merge(
    ops[['gvkey', 'Ticker', 'Filing_Date', 'City', 'operations_state', 'operations_zip']],
    on='gvkey',
    how='left',
)

# ── Ops coverage ──────────────────────────────────────────────────────────────
ops_tickers = ops['Ticker'].nunique()
ops_gvkeys  = ops['gvkey'].nunique()
universe_permnos = universe['permno'].nunique()
print(f"Ops file: {ops_tickers:,} unique tickers ({ops_gvkeys:,} gvkeys) filled with 10-K location data")
print(f"  → {ops_gvkeys / universe_permnos:.1%} of universe permnos matched ({universe_permnos:,} total)")


Ops file: 209 unique tickers (209 gvkeys) filled with 10-K location data
  → 10.0% of universe permnos matched (2,087 total)


In [5]:
# ── Resolve city/state: prefer 10-K operations data, fall back to CRSP HQ ─────
# Operations fields: City, operations_state  (from 10-K Item 2 scan)
# CRSP HQ fields  : city, state             (Compustat registered address)
#
# Rules:
#   1. Both ops city AND ops state present  → use operations city + state
#   2. Only one of ops city / ops state     → fall back to HQ city + HQ state
#   3. Neither                              → fall back to HQ city + HQ state

has_both_ops = merged['City'].notna() & merged['operations_state'].notna()

merged['resolved_city']  = np.where(has_both_ops, merged['City'],  merged['city'])
merged['resolved_state'] = np.where(has_both_ops, merged['operations_state'], merged['state'])

# Normalise for lookup: lowercase city, uppercase state abbreviation
merged['_city_key']  = merged['resolved_city'].str.strip().str.lower()
merged['_state_key'] = merged['resolved_state'].str.strip().str.upper()

# ── Map (city, state) → 5-digit Census county FIPS ───────────────────────────
merged['county_fips'] = merged.apply(
    lambda r: city_county.get((r['_city_key'], r['_state_key'])),
    axis=1,
)

# Drop temp keys
merged.drop(columns=['_city_key', '_state_key'], inplace=True)

# ── Coverage report ────────────────────────────────────────────────────────────
has_ops  = has_both_ops
has_fips = merged['county_fips'].notna()

print(f"\nSample:")
print(merged[['permno','date','City','operations_state','city','state',
              'resolved_city','resolved_state','county_fips']]
      .dropna(subset=['county_fips']).head(5).to_string(index=False))


Sample:
 permno       date City operations_state       city state resolved_city resolved_state county_fips
  10001 2011-04-29  NaN              NaN Naperville    IL    Naperville             IL       17043
  10001 2011-05-31  NaN              NaN Naperville    IL    Naperville             IL       17043
  10001 2011-06-30  NaN              NaN Naperville    IL    Naperville             IL       17043
  10001 2011-07-29  NaN              NaN Naperville    IL    Naperville             IL       17043
  10001 2011-08-31  NaN              NaN Naperville    IL    Naperville             IL       17043


In [6]:
# ── Load precipitation and temperature county anomalies ───────────────────────
prec = pd.read_parquet(os.path.join(intermediate_dir, 'prec_county_anomaly.parquet'),
                       columns=['geo_id', 'year', 'month_num', 'z_score', 'anomaly'])
temp = pd.read_parquet(os.path.join(intermediate_dir, 'temp_county_anomaly.parquet'),
                       columns=['geo_id', 'year', 'month_num', 'z_score', 'anomaly'])

prec = prec.rename(columns={'z_score': 'rainfall_z_score', 'anomaly': 'rainfall_anomaly'})
temp = temp.rename(columns={'z_score': 'temp_z_score',     'anomaly': 'temp_anomaly'})

# ── Standardise geo_id to zero-padded 5-char string (matches county_fips) ────
for df, name in [(prec, 'prec'), (temp, 'temp')]:
    df['geo_id'] = df['geo_id'].astype(str).str.strip().str.zfill(5)
    lens = df['geo_id'].str.len().value_counts()
    print(f"{name} geo_id dtype={df['geo_id'].dtype}  length dist: {lens.to_dict()}  sample: {df['geo_id'].head(3).tolist()}")

# ── Verify key alignment before merging ───────────────────────────────────────
panel_fips  = set(merged['county_fips'].dropna().unique())
prec_gids   = set(prec['geo_id'].unique())
overlap     = panel_fips & prec_gids
print(f"\npanel county_fips:  {len(panel_fips):,} unique counties (sample: {sorted(panel_fips)[:3]})")
print(f"prec geo_id:        {len(prec_gids):,} unique counties  (sample: {sorted(prec_gids)[:3]})")
print(f"Overlap:            {len(overlap):,}  ({len(overlap)/len(panel_fips):.1%} of panel covered)")

# ── Build merge keys on merged df ─────────────────────────────────────────────
merged['_year']      = merged['date'].dt.year
merged['_month_num'] = merged['date'].dt.month

# ── Merge precipitation ────────────────────────────────────────────────────────
merged = merged.merge(
    prec,
    left_on  = ['county_fips', '_year', '_month_num'],
    right_on = ['geo_id',      'year',  'month_num'],
    how='left',
).drop(columns=['geo_id', 'year', 'month_num'])

# ── Merge temperature ──────────────────────────────────────────────────────────
merged = merged.merge(
    temp,
    left_on  = ['county_fips', '_year', '_month_num'],
    right_on = ['geo_id',      'year',  'month_num'],
    how='left',
).drop(columns=['geo_id', 'year', 'month_num'])

merged.drop(columns=['_year', '_month_num'], inplace=True)

# ── Coverage report ────────────────────────────────────────────────────────────
print(f"\nRows: {len(merged):,}")
print(f"rainfall_z_score fill : {merged['rainfall_z_score'].notna().mean():.1%}")
print(f"temp_z_score fill     : {merged['temp_z_score'].notna().mean():.1%}")
print(f"rainfall_anomaly dist :\n{merged['rainfall_anomaly'].value_counts(dropna=False).head()}")
print(f"temp_anomaly dist     :\n{merged['temp_anomaly'].value_counts(dropna=False).head()}")


prec geo_id dtype=str  length dist: {5: 4960368}  sample: ['01001', '01001', '01001']
temp geo_id dtype=str  length dist: {5: 4960368}  sample: ['01001', '01001', '01001']

panel county_fips:  557 unique counties (sample: ['01003', '01069', '01073'])
prec geo_id:        3,142 unique counties  (sample: ['01001', '01003', '01005'])
Overlap:            549  (98.6% of panel covered)

Rows: 347,798
rainfall_z_score fill : 88.8%
temp_z_score fill     : 88.8%
rainfall_anomaly dist :
rainfall_anomaly
NaN      331962
above     15624
below       212
Name: count, dtype: int64
temp_anomaly dist     :
temp_anomaly
NaN      333332
above     10800
below      3666
Name: count, dtype: int64


In [7]:
# ── Load solar irradiance anomalies (state × year × month) ───────────────────
solar = pd.read_parquet(os.path.join(intermediate_dir, 'solar_panel.parquet'),
                        columns=['State_Abbr', 'year', 'month_num', 'DNI_z', 'DNI_anomaly'])

solar = solar.rename(columns={'DNI_z': 'solar_z_score', 'DNI_anomaly': 'solar_anomaly'})

# ── Merge on resolved_state + year + month ────────────────────────────────────
merged['_year']      = merged['date'].dt.year
merged['_month_num'] = merged['date'].dt.month

merged = merged.merge(
    solar,
    left_on  = ['resolved_state', '_year', '_month_num'],
    right_on = ['State_Abbr',     'year',  'month_num'],
    how='left',
).drop(columns=['State_Abbr', 'year', 'month_num'])

merged.drop(columns=['_year', '_month_num'], inplace=True)

In [8]:
# ── Drop unnecessary columns ──────────────────────────────────────────────────
# Dropped:
#   prc, shrout, cfacpr  — raw price/shares (mktcap already computed)
#   shrcd                — share code, already filtered to 10/11
#   ticker               — CRSP ticker, tic (Compustat) kept instead
#   datadate             — Compustat fiscal year-end, internal merge key
#   cusip                — redundant identifier
#   conm                 — Compustat name, comnam (CRSP) kept instead
#   sich, naicsh         — 64% / 70% fill; siccd used for sector assignment
#   csho, prcc_f         — Compustat duplicates of market data
#   addzip, loc          — zip/country code; county_fips is the geo key
#   Ticker, Filing_Date  — ops metadata, not needed downstream
#   City, operations_state, operations_zip — raw ops fields; resolved_* kept

DROP_COLS = [
    # raw CRSP price/shares
    'prc', 'shrout', 'cfacpr', 'shrcd',
    # redundant identifiers / names
    'ticker', 'cusip', 'conm',
    # internal merge artifacts
    'datadate',
    # low-fill or redundant industry codes
    'sich', 'naicsh',
    # Compustat duplicates of market data
    'csho', 'prcc_f',
    # geography superseded by county_fips
    'addzip', 'loc',
    # raw ops fields superseded by resolved_*
    'Ticker', 'Filing_Date', 'City', 'operations_state', 'operations_zip',
]

merged.drop(columns=[c for c in DROP_COLS if c in merged.columns], inplace=True)

print(f"Columns kept ({len(merged.columns)}): {merged.columns.tolist()}")
print(f"Rows: {len(merged):,}")
print(merged.head(3).to_string())

Columns kept (25): ['permno', 'date', 'ret', 'exchcd', 'siccd', 'comnam', 'mktcap', 'index', 'atmos_sector', 'gvkey', 'tic', 'at', 'sale', 'ceq', 'state', 'city', 'resolved_city', 'resolved_state', 'county_fips', 'rainfall_z_score', 'rainfall_anomaly', 'temp_z_score', 'temp_anomaly', 'solar_z_score', 'solar_anomaly']
Rows: 347,798
   permno       date       ret  exchcd  siccd           comnam    mktcap         index         atmos_sector   gvkey   tic       at  sale     ceq state        city resolved_city resolved_state county_fips  rainfall_z_score rainfall_anomaly  temp_z_score temp_anomaly  solar_z_score solar_anomaly
0   10001 2011-04-29 -0.038789       2   4925  GAS NATURAL INC  91535.73  Russell 2000  Utilities & Telecom  012994  EGAS  137.728  91.5  73.701    IL  Naperville    Naperville             IL       17043          0.910570              NaN     -0.165591          NaN       0.076258           NaN
1   10001 2011-05-31   0.02805       2   4925  GAS NATURAL INC   93736.5  Rus

In [9]:
# ── Collapse multi-location rows → one row per permno × date ─────────────────
#
# z-scores → equal-weighted mean across all operating locations
#            (z-scores share a common N(0,1) scale so averaging is valid)
# county_fips → modal county (for HQ-only firms this is just their county)
# n_locations → distinct county_fips values; useful for robustness checks
#
# Anomaly labels are derived AFTER collapsing from the aggregated z-score:
#   z ≥ +2  → "above"   (≈ top 2.3% of historical distribution)
#   z ≤ −2  → "below"   (≈ bottom 2.3%)
#   else    → NaN       (normal — not flagged)

STABLE_COLS = [
    'permno', 'date', 'ret', 'mktcap', 'exchcd', 'siccd',
    'comnam', 'tic', 'index', 'atmos_sector', 'gvkey',
    'at', 'sale', 'ceq', 'state', 'city',
]

Z_COLS = ['rainfall_z_score', 'temp_z_score', 'solar_z_score']

# Drop source anomaly labels — we rederive from z-scores below
SOURCE_ANOMALY_COLS = ['rainfall_anomaly', 'temp_anomaly', 'solar_anomaly']

def _mode(s):
    m = s.dropna()
    return m.mode().iloc[0] if len(m) else np.nan

# Count distinct operating counties before collapsing
merged['n_locations'] = (
    merged.groupby(['permno', 'date'], observed=True)['county_fips']
          .transform('nunique')
)

agg_spec = (
    {c: 'mean' for c in Z_COLS}
  | {'county_fips': _mode, 'n_locations': 'first'}
)

panel = (
    merged
    .drop(columns=['resolved_city', 'resolved_state'] + SOURCE_ANOMALY_COLS)
    .groupby(STABLE_COLS, observed=True, dropna=False)
    .agg(agg_spec)
    .reset_index()
)

# ── Derive anomaly labels from collapsed z-scores (±2σ threshold) ────────────
# Use None instead of np.nan to avoid NumPy 2.x DTypePromotionError
# (mixing str and float dtypes in np.where is not allowed in NumPy 2+)
def _label(col):
    return np.where(col >= 2, 'above',
           np.where(col <= -2, 'below', None))

panel['rainfall_anomaly'] = _label(panel['rainfall_z_score'])
panel['temp_anomaly']     = _label(panel['temp_z_score'])
panel['solar_anomaly']    = _label(panel['solar_z_score'])

# ── Diagnostics ───────────────────────────────────────────────────────────────
print(f"Before collapse : {len(merged):,} rows")
print(f"After collapse  : {len(panel):,} rows")
print(f"\nn_locations distribution:")
print(panel['n_locations'].value_counts().sort_index().head(10).to_string())
print(f"\nrainfall_anomaly : {dict(zip(*np.unique(panel['rainfall_anomaly'].dropna(), return_counts=True)))}")
print(f"temp_anomaly     : {dict(zip(*np.unique(panel['temp_anomaly'].dropna(),     return_counts=True)))}")
print(f"solar_anomaly    : {dict(zip(*np.unique(panel['solar_anomaly'].dropna(),    return_counts=True)))}")
print(f"\nColumns ({len(panel.columns)}): {panel.columns.tolist()}")

Before collapse : 347,798 rows
After collapse  : 215,342 rows

n_locations distribution:
n_locations
0     33837
1    155644
2     10103
3      4312
4      1124
5      2678
6      1536
7       614
8       365
9       211

rainfall_anomaly : {'above': np.int64(8006), 'below': np.int64(117)}
temp_anomaly     : {'above': np.int64(4337), 'below': np.int64(2245)}
solar_anomaly    : {'above': np.int64(1530), 'below': np.int64(1207)}

Columns (24): ['permno', 'date', 'ret', 'mktcap', 'exchcd', 'siccd', 'comnam', 'tic', 'index', 'atmos_sector', 'gvkey', 'at', 'sale', 'ceq', 'state', 'city', 'rainfall_z_score', 'temp_z_score', 'solar_z_score', 'county_fips', 'n_locations', 'rainfall_anomaly', 'temp_anomaly', 'solar_anomaly']


In [10]:
# ── Save full panel (NaNs preserved; regressions drop missing rows per model) ──
print(f"panel : {len(panel):,} rows  |  {panel['permno'].nunique():,} firms")
print(f"  rainfall_z_score fill : {panel['rainfall_z_score'].notna().mean():.1%}")
print(f"  temp_z_score fill     : {panel['temp_z_score'].notna().mean():.1%}")
print(f"  solar_z_score fill    : {panel['solar_z_score'].notna().mean():.1%}")

panel.to_parquet(os.path.join(intermediate_dir, 'panel.parquet'), index=False)
print(f"\nSaved: panel.parquet → {len(panel):,} rows")

panel : 215,342 rows  |  2,087 firms
  rainfall_z_score fill : 82.1%
  temp_z_score fill     : 82.1%
  solar_z_score fill    : 37.6%

Saved: panel.parquet → 215,342 rows
